# SPINE-GPE v7 — Fase 0 — Colab v7.0.3
Versão **local-first**: detecta automaticamente os arquivos enviados em `SPINE-GPEv7/data_pnadc`, registra todos os uploads `data_*`, baixa documentação e fontes oficiais, e materializa separadamente os suplementos diretos de plataformas 2022T4 e 2024T3.


In [1]:
from google.colab import drive
from pathlib import Path
import os, sys, subprocess, json
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7')
DATA_PNADC = ROOT / 'data_pnadc'
ROOT.mkdir(parents=True, exist_ok=True)
DATA_PNADC.mkdir(parents=True, exist_ok=True)
os.environ['SPINE_GPE_ROOT'] = str(ROOT)
print('ROOT =', ROOT)
print('DATA_PNADC =', DATA_PNADC)


Mounted at /content/drive
ROOT = /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7
DATA_PNADC = /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/data_pnadc


## 1. Confirmar os uploads PNADc


In [2]:
files_found = sorted([p for p in DATA_PNADC.rglob('*') if p.is_file()])
for p in files_found:
    print(f'{p.relative_to(ROOT)} | {p.stat().st_size/1024**2:,.1f} MiB')
required = {'PNADC_042022.txt', 'PNADC_032024.txt'}
present = {p.name for p in files_found}
missing = required - present
assert not missing, f'Arquivos ausentes em data_pnadc: {sorted(missing)}'


data_pnadc/PNADC_032024.txt | 1,592.3 MiB
data_pnadc/PNADC_042022.txt | 1,586.7 MiB


## 2. Disponibilizar o script v7.0.3
Faça upload do `.py` para `/content` ou copie-o para a raiz do projeto no Drive.


In [5]:
candidates = [
    Path('/content/SPINE_GPEv7_FASE0_v7.0.3.py'),
    ROOT / 'SPINE_GPEv7_FASE0_v7.0.3.py',
    ROOT / 'scripts' / 'SPINE_GPEv7_FASE0_v7.0.3.py',
]
SCRIPT = next((p for p in candidates if p.exists()), None)
if SCRIPT is None:
    from google.colab import files
    uploaded = files.upload()
    names = [n for n in uploaded if n.endswith('.py')]
    assert names, 'Envie SPINE_GPEv7_FASE0_v7.0.3.py'
    SCRIPT = Path('/content') / names[0]
print('SCRIPT =', SCRIPT)


Saving SPINE_GPEv7_FASE0_v7.0.3.py to SPINE_GPEv7_FASE0_v7.0.3.py
SCRIPT = /content/SPINE_GPEv7_FASE0_v7.0.3.py


## 3. Instalar dependências


In [3]:
packages = [
 'requests>=2.31','beautifulsoup4>=4.12','lxml>=5.0','pandas>=2.1','numpy>=1.26',
 'pyarrow>=15','polars>=0.20','charset-normalizer>=3.3','py7zr>=0.21',
 'openpyxl>=3.1','xlrd>=2.0','odfpy>=1.4','pypdf>=4.0','PyYAML>=6.0',
 'tqdm>=4.66','rich>=13.7','tenacity>=8.2','fsspec>=2024.2',
 'osmnx>=1.9','geopandas>=0.14','shapely>=2.0'
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'requests>=2.31', 'beautifulsoup4>=4.12', 'lxml>=5.0', 'pandas>=2.1', 'numpy>=1.26', 'pyarrow>=15', 'polars>=0.20', 'charset-normalizer>=3.3', 'py7zr>=0.21', 'openpyxl>=3.1', 'xlrd>=2.0', 'odfpy>=1.4', 'pypdf>=4.0', 'PyYAML>=6.0', 'tqdm>=4.66', 'rich>=13.7', 'tenacity>=8.2', 'fsspec>=2024.2', 'osmnx>=1.9', 'geopandas>=0.14', 'shapely>=2.0'], returncode=0)

## 4. Executar a Fase 0 em modo core
O modo `core` usa os uploads trimestrais, baixa os layouts e os suplementos anuais concentrados necessários para validar `S140093`, além das fontes públicas essenciais.


In [6]:
cmd = [
    sys.executable, str(SCRIPT),
    '--root', str(ROOT),
    '--local-data-dir', str(DATA_PNADC),
    '--mode', 'core',
]
print(' '.join(cmd))
result = subprocess.run(cmd, check=False)
print('Exit code:', result.returncode)


/usr/bin/python3 /content/SPINE_GPEv7_FASE0_v7.0.3.py --root /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7 --local-data-dir /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/data_pnadc --mode core
Exit code: 0


## 5. Ler o lock e o relatório


In [7]:
lock_path = ROOT / '00_admin' / 'PHASE0_LOCK.json'
report_path = ROOT / '00_admin' / 'reports' / 'PHASE0_REPORT_LATEST.md'
print(json.dumps(json.loads(lock_path.read_text(encoding='utf-8')), indent=2, ensure_ascii=False))
print('\n--- RELATÓRIO (final) ---\n')
report = report_path.read_text(encoding='utf-8', errors='replace')
print(report[-16000:])


{
  "run_id": "20260720T020435Z",
  "script_version": "7.0.3-phase0-local-first-supplement-safe",
  "schema_version": "spine-gpe-v7-schema-1.1.0",
  "status": "RELEASED",
  "critical_failures": [],
  "report": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/reports/PHASE0_REPORT_20260720T020435Z.md",
  "created_at_utc": "2026-07-20T02:29:07.825981+00:00"
}

--- RELATÓRIO (final) ---

e_velocidade_2022 | `/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/30_recife_ckan/velocidade_2022/Lombadas_2022_-_Setembro_-_Quantitativo_das_Vias_por_Velocidade_Média.csv` | 0.017 | `b45709feec411b6c…` | 178636 |  |
| recife_velocidade_2022 | `/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/30_recife_ckan/velocidade_2022/Fotossensores_2022_-_Outubro_-_Quantitativo_das_Vias_por_Velocidade_Média.csv` | 0.028 | `9be34572d20ba49a…` | 332473 |  |
| recife_velocidade_2022 | `/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/01_raw/30_recife_ckan/velocidade_2022/Lo